In [1]:
#!/usr/bin/env python3
# coding: utf-8
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import glob
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import csv
import os
import datetime

# ============================================================================
# HELPERS
# ============================================================================
def setup_device():
    if torch.cuda.is_available():
        device = torch.device('cuda')
        device_name = torch.cuda.get_device_name(0)
        gpu_memory_total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print("\n" + "="*70)
        print("CUDA DEVICE DETECTED - GPU ACCELERATION ENABLED")
        print("="*70)
        print(f"✓ Device: {device_name}")
        print(f"✓ Compute Capability: {torch.cuda.get_device_capability(0)}")
        print(f"✓ Total Memory: {gpu_memory_total:.2f} GB")
        print(f"✓ PyTorch Version: {torch.__version__}")
        print(f"✓ CUDA Version: {torch.version.cuda}")
        print("="*70 + "\n")
        return device, f"{device_name} (GPU)"
    else:
        device = torch.device('cpu')
        print("\n" + "="*70)
        print("CUDA NOT AVAILABLE - USING CPU")
        print("="*70)
        print("⚠ GPU not detected. Using CPU for training (slower)")
        print(f"✓ CPU Cores: {os.cpu_count()}")
        print("="*70 + "\n")
        return device, "CPU"

def extract_id_from_filename(filename):
    basename = os.path.basename(filename)
    return os.path.splitext(basename)[0]

def create_labels_from_ids(file_list, false_ids_list=None):
    if false_ids_list is None:
        false_ids_list = []
    ids = [extract_id_from_filename(f) for f in file_list]
    labels = np.array([0 if id_str in false_ids_list else 1 for id_str in ids])
    return labels, ids

def pad_or_trim_spectrum(spec, target_len, pad_value=0.0):
    """
    Pad (with pad_value) or trim spectrum to target_len.
    pad_value may be np.nan when desired.
    """
    spec = np.asarray(spec, dtype=float)
    if len(spec) < target_len:
        pad_len = target_len - len(spec)
        return np.concatenate([spec, np.full(pad_len, pad_value, dtype=float)])
    elif len(spec) > target_len:
        return spec[:target_len]
    else:
        return spec

# ============================================================================
# CONFIG & FALSE LISTS (kept from notebook)
# ============================================================================
train_false_candidates = [
    'HVC225.14+6.52+103', 'HVC228.83+13.02+110', 'HVC228.40+12.31+110', 'HVC224.92+6.10+101',
    'HVC199.71-29.60-249', 'HVC202.58-26.55-80', 'HVC199.50-31.25-62', 'HVC199.27-31.79-57',
    'HVC197.79-33.79-57', 'HVC197.20-34.63-57', 'HVC196.17-36.17-75', 'HVC196.58-35.58-89',
    'HVC197.55-34.15-69', 'HVC199.10-31.75-58'
]

all_false_hvc_candidates = [
    # partial list; use full from your notebook if needed
    'HVC209.05-24.89+89', 'HVC205.30-31.02+84', 'HVC208.94-24.49+86',
    'HVC202.90-34.43+86', 'HVC201.79-35.94+89', 'HVC203.59-33.19+85',
    'HVC224.97+16.83-250', 'HVC234.71+14.63-212', 'HVC233.48+21.12-85',
    'HVC235.91+24.73-69', 'HVC228.50+28.98-71'
]


In [2]:

# ============================================================================
# 2D SECTION (unchanged behavior)
# ============================================================================
print("\n" + "="*70)
print("HVC CATALOG CNN MODEL - 2D & 1D COMBINED CLASSIFICATION")
print("="*70)
print(f"Training Date: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"User: Firestar-Reimu-Pro")



HVC CATALOG CNN MODEL - 2D & 1D COMBINED CLASSIFICATION
Training Date: 2025-11-22 17:34:23
User: Firestar-Reimu-Pro


In [3]:

device, device_name = setup_device()

print("="*70)
print("LOADING TRAINING DATA (2D IMAGES)")
print("="*70)
file_list_2d_train = sorted(glob.glob('./Test/2D_morph/*.txt'))
arrays_2d_train = [np.loadtxt(fname) for fname in file_list_2d_train]
labels_2d_train, ids_2d_train = create_labels_from_ids(file_list_2d_train, train_false_candidates)
print(f"✓ Total 2D training samples: {len(arrays_2d_train)}")

print("\n" + "="*70)
print("LOADING TEST DATA (2D IMAGES)")
print("="*70)
file_list_2d_test = sorted(glob.glob('./2D_morph/*.txt'))
arrays_2d_test = [np.loadtxt(fname) for fname in file_list_2d_test]
labels_2d_test, ids_2d_test = create_labels_from_ids(file_list_2d_test, all_false_hvc_candidates)
print(f"✓ Total 2D test samples: {len(arrays_2d_test)}")

class VariableShapeMorphDataset(Dataset):
    def __init__(self, arrays, labels):
        self.arrays = arrays
        self.labels = labels
    def __len__(self):
        return len(self.arrays)
    def __getitem__(self, idx):
        x = torch.tensor(self.arrays[idx], dtype=torch.float32).unsqueeze(0)
        y = torch.tensor(self.labels[idx], dtype=torch.float32)
        return x, y

train_dataset_2d = VariableShapeMorphDataset(arrays_2d_train, labels_2d_train)
test_dataset_2d = VariableShapeMorphDataset(arrays_2d_test, labels_2d_test)
train_loader_2d = DataLoader(train_dataset_2d, batch_size=1, shuffle=True)
test_loader_2d = DataLoader(test_dataset_2d, batch_size=1, shuffle=False)

class VariableShapeCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, 3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(16, 1)
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        x = torch.sigmoid(self.fc(x)).squeeze(-1)
        return x

model_2d = VariableShapeCNN().to(device)
optimizer_2d = torch.optim.Adam(model_2d.parameters(), lr=1e-3)
loss_fn_2d = nn.BCELoss()

num_epochs_2d = 50
print("\n" + "="*70)
print(f"TRAINING 2D MODEL ({num_epochs_2d} epochs)")
print("="*70)
for epoch in range(num_epochs_2d):
    model_2d.train()
    losses = []
    for x, y in train_loader_2d:
        x, y = x.to(device), y.to(device)
        out = model_2d(x)
        loss = loss_fn_2d(out.view(-1), y.view(-1))
        optimizer_2d.zero_grad()
        loss.backward()
        optimizer_2d.step()
        losses.append(loss.item())
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:3d}/{num_epochs_2d}, avg train loss: {np.mean(losses):.6f}')

model_2d.eval()
preds_2d = []
trues_2d = []
with torch.no_grad():
    for x, y in test_loader_2d:
        x, y = x.to(device), y.to(device)
        out = model_2d(x)
        pred = (out.view(-1) > 0.5).int().item()
        preds_2d.append(pred)
        trues_2d.append(y.item())
print('\n=== 2D Model Test Results ===')
acc_2d = accuracy_score(trues_2d, preds_2d)

torch.save(model_2d.state_dict(), "cnn_2d_morph.pt")
print("2D Model saved to: cnn_2d_morph.pt")



CUDA DEVICE DETECTED - GPU ACCELERATION ENABLED
✓ Device: NVIDIA GeForce RTX 4060 Ti
✓ Compute Capability: (8, 9)
✓ Total Memory: 8.15 GB
✓ PyTorch Version: 2.7.1
✓ CUDA Version: 12.9

LOADING TRAINING DATA (2D IMAGES)
✓ Total 2D training samples: 38

LOADING TEST DATA (2D IMAGES)
✓ Total 2D test samples: 101

TRAINING 2D MODEL (50 epochs)
Epoch   5/50, avg train loss: 0.539083
Epoch  10/50, avg train loss: 0.372018
Epoch  15/50, avg train loss: 0.280445
Epoch  20/50, avg train loss: 0.241072
Epoch  25/50, avg train loss: 0.252019
Epoch  30/50, avg train loss: 0.187201
Epoch  35/50, avg train loss: 0.156378
Epoch  40/50, avg train loss: 0.163402
Epoch  45/50, avg train loss: 0.132197
Epoch  50/50, avg train loss: 0.113848

=== 2D Model Test Results ===
2D Model saved to: cnn_2d_morph.pt


In [4]:

# ============================================================================
# 1D SECTION - MODIFIED: NaN pad + mask channels + larger kernel + BN + Dropout
# ============================================================================
print("\n" + "="*70)
print("LOADING TRAINING DATA (1D SPECTRUM)")
print("="*70)

# load training spectra and references
spectrum_files_train = sorted(glob.glob('./Test/crafts_spectrum/*.txt'))
ref_spectrum_files_train = sorted(glob.glob('./Test/hi4pi_spectrum/*.txt'))
spectra_1d_train = [np.loadtxt(fname) for fname in spectrum_files_train]
spectra_1d_ref_train = [np.loadtxt(fname) for fname in ref_spectrum_files_train]
labels_1d_train, ids_1d_train = create_labels_from_ids(spectrum_files_train, train_false_candidates)

print(f"✓ Total 1D training samples: {len(spectra_1d_train)}")

print("\n" + "="*70)
print("LOADING TEST DATA (1D SPECTRUM)")
print("="*70)

spectrum_files_test = sorted(glob.glob('./crafts_spectrum/*.txt'))
ref_spectrum_files_test = sorted(glob.glob('./hi4pi_spectrum/*.txt'))
spectra_1d_test = [np.loadtxt(fname) for fname in spectrum_files_test]
spectra_1d_ref_test = [np.loadtxt(fname) for fname in ref_spectrum_files_test]
labels_1d_test, ids_1d_test = create_labels_from_ids(spectrum_files_test, all_false_hvc_candidates)

print(f"✓ Total 1D test samples: {len(spectra_1d_test)}")

# compute global maximum length across all four lists and pad with np.nan
all_lengths = []
for lst in (spectra_1d_train, spectra_1d_ref_train, spectra_1d_test, spectra_1d_ref_test):
    if lst:
        all_lengths.extend([len(s) for s in lst])
target_length = max(all_lengths) if all_lengths else 0
print(f"Target length (global max) = {target_length}")

# pad with np.nan to allow mask-based handling
spectra_1d_train = [pad_or_trim_spectrum(s, target_length, pad_value=np.nan) for s in spectra_1d_train]
spectra_1d_ref_train = [pad_or_trim_spectrum(s, target_length, pad_value=np.nan) for s in spectra_1d_ref_train]
spectra_1d_test = [pad_or_trim_spectrum(s, target_length, pad_value=np.nan) for s in spectra_1d_test]
spectra_1d_ref_test = [pad_or_trim_spectrum(s, target_length, pad_value=np.nan) for s in spectra_1d_ref_test]

print("All spectra padded/trimmed to target_length with np.nan (masks will be provided).")

class Spectrum1DDataset(Dataset):
    """
    Now returns 4-channel input:
      0: spectrum (standardized, NaNs replaced with 0 after normalization)
      1: reference (standardized, NaNs replaced with 0 after normalization)
      2: mask_spec (1.0 for valid data, 0.0 for padded)
      3: mask_ref
    Standardization is per-sample using nanmean / nanstd.
    """
    def __init__(self, spectra, spectra_ref, labels):
        assert len(spectra) == len(spectra_ref), "spectra and spectra_ref must match in length"
        self.spectra = spectra
        self.spectra_ref = spectra_ref
        self.labels = labels

    def __len__(self):
        return len(self.spectra)

    def __getitem__(self, idx):
        spec = np.asarray(self.spectra[idx], dtype=float)
        ref = np.asarray(self.spectra_ref[idx], dtype=float)

        mask_spec = (~np.isnan(spec)).astype(float)
        mask_ref = (~np.isnan(ref)).astype(float)

        # per-sample normalization ignoring NaNs
        if np.any(mask_spec):
            m_spec = np.nanmean(spec)
            s_spec = np.nanstd(spec)
            if s_spec == 0 or np.isnan(s_spec):
                s_spec = 1.0
            spec_norm = (spec - m_spec) / s_spec
        else:
            spec_norm = np.zeros_like(spec, dtype=float)

        if np.any(mask_ref):
            m_ref = np.nanmean(ref)
            s_ref = np.nanstd(ref)
            if s_ref == 0 or np.isnan(s_ref):
                s_ref = 1.0
            ref_norm = (ref - m_ref) / s_ref
        else:
            ref_norm = np.zeros_like(ref, dtype=float)

        # replace NaN (the padded positions) with 0 after normalization
        spec_norm = np.nan_to_num(spec_norm, nan=0.0)
        ref_norm = np.nan_to_num(ref_norm, nan=0.0)

        x = np.stack([spec_norm, ref_norm, mask_spec, mask_ref], axis=0).astype(np.float32)
        y = np.float32(self.labels[idx])
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

train_dataset_1d = Spectrum1DDataset(spectra_1d_train, spectra_1d_ref_train, labels_1d_train)
test_dataset_1d = Spectrum1DDataset(spectra_1d_test, spectra_1d_ref_test, labels_1d_test)
train_loader_1d = DataLoader(train_dataset_1d, batch_size=1, shuffle=True)
test_loader_1d = DataLoader(test_dataset_1d, batch_size=1, shuffle=False)



LOADING TRAINING DATA (1D SPECTRUM)
✓ Total 1D training samples: 38

LOADING TEST DATA (1D SPECTRUM)
✓ Total 1D test samples: 101
Target length (global max) = 79
All spectra padded/trimmed to target_length with np.nan (masks will be provided).


In [29]:
class ResidualDilatedBlock(nn.Module):
    """
    小的残差块：Conv -> BN -> ReLU -> Conv(dilated) -> BN -> add -> ReLU
    使用较小 kernel + dilation 来扩大感受野
    """
    def __init__(self, in_ch, out_ch, kernel=11, dilation=1, dropout=0.1):
        super().__init__()
        mid_ch = out_ch
        padding = (kernel // 2) * dilation
        self.conv1 = nn.Conv1d(in_ch, mid_ch, kernel_size=kernel, padding=padding, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(mid_ch)
        self.conv2 = nn.Conv1d(mid_ch, out_ch, kernel_size=kernel, padding=padding, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.need_proj = (in_ch != out_ch)
        if self.need_proj:
            self.proj = nn.Conv1d(in_ch, out_ch, kernel_size=1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        residual = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.dropout(out)
        if self.need_proj:
            residual = self.proj(residual)
        out = F.relu(out + residual)
        return out

class CNN1D_Improved(nn.Module):
    """
    输入: channels = 4 (spec_norm, ref_norm, mask_spec, mask_ref)
    结构策略:
      - 初始 1x1 投影到较多通道
      - 多个 ResidualDilatedBlock，dilation 随层增加 (1,2,4)，保证覆盖宽峰但保持参数量
      - GAP -> FC 输出 logits (不带 sigmoid)
    """
    def __init__(self, length, input_ch=4, base_ch=48, dropout=0.25):
        super().__init__()
        self.input_proj = nn.Conv1d(input_ch, base_ch, kernel_size=1)
        # 三个残差层，dilation 的选择使感受野快速扩大
        self.block1 = ResidualDilatedBlock(base_ch, base_ch, kernel=11, dilation=1, dropout=dropout)
        self.block2 = ResidualDilatedBlock(base_ch, base_ch, kernel=11, dilation=2, dropout=dropout)
        self.block3 = ResidualDilatedBlock(base_ch, base_ch, kernel=9, dilation=4, dropout=dropout)
        # optional extra block for more capacity
        self.block4 = ResidualDilatedBlock(base_ch, base_ch*2, kernel=7, dilation=2, dropout=dropout)
        self.bn_final = nn.BatchNorm1d(base_ch*2)
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(base_ch*2, 1)

        # init
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.0)

    def forward(self, x):
        # x: (B, 4, L)
        x = F.relu(self.input_proj(x))        # (B, base_ch, L)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = F.relu(self.bn_final(x))
        x = self.gap(x)                       # (B, C, 1)
        x = x.view(x.size(0), -1)             # (B, C)
        logits = self.fc(x).squeeze(-1)       # (B,)
        return logits

In [30]:

# ============================================================================
# CNN1D: larger kernels (~peak width 30), more channels, BN, Dropout
# ============================================================================
class CNN1D(nn.Module):
    """
    Input channels = 4 (spec, ref, mask_spec, mask_ref)
    """
    def __init__(self, length):
        super().__init__()
        # conv1 large kernel ~ peak width (31) to capture full peak
        self.conv1 = nn.Conv1d(4, 32, kernel_size=31, padding=15)
        self.bn1 = nn.BatchNorm1d(32)
        self.pool = nn.MaxPool1d(2)
        # second conv smaller kernel for detail
        self.conv2 = nn.Conv1d(32, 64, kernel_size=11, padding=5)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool2 = nn.MaxPool1d(2)
        self.dropout = nn.Dropout(0.3)
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(64, 1)  # produces logits

        # init
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.0)

    def forward(self, x):
        # x shape: (B, 4, L)
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool2(x)
        x = self.dropout(x)
        x = self.gap(x)  # (B, C, 1)
        x = x.view(x.size(0), -1)  # (B, C)
        logits = self.fc(x).squeeze(-1)  # (B,)
        return logits

spectrum_length_train = target_length
model_1d = CNN1D_Improved(spectrum_length_train).to(device)

# optimizer with a small weight decay to improve generalization
optimizer_1d = torch.optim.Adam(model_1d.parameters(), lr=5e-4, weight_decay=1e-5)
# use BCEWithLogitsLoss for numeric stability
loss_fn_1d = nn.BCEWithLogitsLoss()

print(f"\n1D Model (modified) created and moved to device: {device_name}")
print(f"Total 1D model parameters: {sum(p.numel() for p in model_1d.parameters()):,}")



1D Model (modified) created and moved to device: NVIDIA GeForce RTX 4060 Ti (GPU)
Total 1D model parameters: 246,289


In [31]:

# ============================================================================
# TRAIN 1D (epochs unchanged)
# ============================================================================
num_epochs_1d = 100
print("\n" + "="*70)
print(f"TRAINING 1D MODEL ({num_epochs_1d} epochs)")
print("="*70)

for epoch in range(num_epochs_1d):
    model_1d.train()
    losses = []
    for x, y in train_loader_1d:
        x, y = x.to(device), y.to(device)
        logits = model_1d(x)  # logits
        loss = loss_fn_1d(logits.view(-1), y.view(-1))
        optimizer_1d.zero_grad()
        loss.backward()
        optimizer_1d.step()
        losses.append(loss.item())
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:3d}/{num_epochs_1d}, avg train loss: {np.mean(losses):.6f}')

# ============================================================================
# EVAL 1D
# ============================================================================
print("\n" + "="*70)
print("EVALUATING 1D MODEL ON TEST SET")
print("="*70)

model_1d.eval()
preds_1d = []
trues_1d = []
with torch.no_grad():
    for x, y in test_loader_1d:
        x, y = x.to(device), y.to(device)
        logits = model_1d(x)
        probs = torch.sigmoid(logits)
        pred = (probs.view(-1) > 0.5).int().item()
        preds_1d.append(pred)
        trues_1d.append(int(y.item()))

print('\n=== 1D Model Test Results ===')
acc_1d = accuracy_score(trues_1d, preds_1d)
prec_1d = precision_score(trues_1d, preds_1d, zero_division=0)
recall_1d = recall_score(trues_1d, preds_1d, zero_division=0)
f1_1d = f1_score(trues_1d, preds_1d, zero_division=0)
print('Test Accuracy:', acc_1d)
print('Test Precision:', prec_1d)
print('Test Recall:', recall_1d)
print('Test F1:', f1_1d)

torch.save(model_1d.state_dict(), "cnn_1d_spectrum.pt")
print("\n1D Model saved to: cnn_1d_spectrum.pt")

# ============================================================================
# Combined predictions, CSV output, statistics, and final cleanup
# (kept same as before; omitted here for brevity but should be the same)
# ============================================================================
# (You can re-use the PRIOR combined/prediction/statistics code from your notebook
#  — it remains compatible because preds_1d/trues_1d shapes are unchanged.)


TRAINING 1D MODEL (100 epochs)
Epoch   5/100, avg train loss: 0.548827
Epoch  10/100, avg train loss: 0.361934
Epoch  15/100, avg train loss: 0.233998
Epoch  20/100, avg train loss: 0.134456
Epoch  25/100, avg train loss: 0.076297
Epoch  30/100, avg train loss: 0.053351
Epoch  35/100, avg train loss: 0.037269
Epoch  40/100, avg train loss: 0.027184
Epoch  45/100, avg train loss: 0.020840
Epoch  50/100, avg train loss: 0.016228
Epoch  55/100, avg train loss: 0.013057
Epoch  60/100, avg train loss: 0.010231
Epoch  65/100, avg train loss: 0.008490
Epoch  70/100, avg train loss: 0.007084
Epoch  75/100, avg train loss: 0.005951
Epoch  80/100, avg train loss: 0.005038
Epoch  85/100, avg train loss: 0.004185
Epoch  90/100, avg train loss: 0.003597
Epoch  95/100, avg train loss: 0.003133
Epoch 100/100, avg train loss: 0.002718

EVALUATING 1D MODEL ON TEST SET

=== 1D Model Test Results ===
Test Accuracy: 0.32673267326732675
Test Precision: 1.0
Test Recall: 0.24444444444444444
Test F1: 0.39285

In [32]:
# ============================================================================
# SECTION 3: COMBINED PREDICTIONS (2D AND 1D)
# ============================================================================

print("\n" + "="*70)
print("GENERATING COMBINED PREDICTIONS")
print("="*70)

# Combine 2D and 1D predictions
# Only mark as True (signal) if BOTH 2D and 1D predict True
# First ensure both test sets have matching IDs
assert ids_2d_test == ids_1d_test, "2D and 1D test sets must have matching IDs"

combined_preds = []
combined_labels = []
combined_ids = []

for id_str, pred_2d, pred_1d, true_label in zip(ids_2d_test, preds_2d, preds_1d, trues_2d):
    # Combined prediction: True only if both 2D AND 1D predict True
    combined_pred = 1 if (pred_2d == 1 and pred_1d == 1) else 0
    combined_preds.append(combined_pred)
    combined_labels.append(true_label)
    combined_ids.append(id_str)

# Calculate combined metrics
print('\n=== Combined Model Test Results ===')
acc_combined = accuracy_score(combined_labels, combined_preds)
prec_combined = precision_score(combined_labels, combined_preds, zero_division=0)
recall_combined = recall_score(combined_labels, combined_preds, zero_division=0)
f1_combined = f1_score(combined_labels, combined_preds, zero_division=0)

print('Test Accuracy:', acc_combined)
print('Test Precision:', prec_combined)
print('Test Recall:', recall_combined)
print('Test F1:', f1_combined)



GENERATING COMBINED PREDICTIONS

=== Combined Model Test Results ===
Test Accuracy: 0.31683168316831684
Test Precision: 1.0
Test Recall: 0.23333333333333334
Test F1: 0.3783783783783784


In [33]:
# ============================================================================
# SECTION 4: OUTPUT DETAILED RESULTS
# ============================================================================

print("\n" + "="*70)
print("SAVING DETAILED PREDICTIONS TO CSV")
print("="*70)

# Define output file name
output_file_combined = "combined_predictions_by_label.csv"

# Write combined predictions grouped by label ID to CSV file
with open(output_file_combined, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    # Write header row
    writer.writerow(['Label_ID', '2D_Prediction', '1D_Prediction', 'Combined_Prediction',
                     'True_Label', '2D_Match', '1D_Match', 'Combined_Match'])

    # Write prediction for each label
    for idx, (id_str, pred_2d, pred_1d, combined_pred, true_label) in enumerate(
            zip(combined_ids, preds_2d, preds_1d, combined_preds, combined_labels)):

        # Convert predictions to string format
        pred_2d_str = 'Signal' if pred_2d == 1 else 'Noise'
        pred_1d_str = 'Signal' if pred_1d == 1 else 'Noise'
        combined_pred_str = 'Signal' if combined_pred == 1 else 'Noise'
        true_label_str = 'Signal' if true_label == 1 else 'Noise'

        # Check if each prediction matches ground truth
        match_2d = 'Yes' if pred_2d == true_label else 'No'
        match_1d = 'Yes' if pred_1d == true_label else 'No'
        match_combined = 'Yes' if combined_pred == true_label else 'No'

        # Write row
        writer.writerow([id_str, pred_2d_str, pred_1d_str, combined_pred_str,
                        true_label_str, match_2d, match_1d, match_combined])

print(f"Combined predictions saved to: {output_file_combined}")

# %%
# Print summary table to console
print("\n" + "="*70)
print("DETAILED PREDICTIONS BY LABEL")
print("="*70)
print(f"{'Label_ID':<25} {'2D':<10} {'1D':<10} {'Combined':<10} {'True':<10} {'Status':<20}")
print("-" * 90)

for id_str, pred_2d, pred_1d, combined_pred, true_label in zip(
        combined_ids, preds_2d, preds_1d, combined_preds, combined_labels):

    pred_2d_str = 'Signal' if pred_2d == 1 else 'Noise'
    pred_1d_str = 'Signal' if pred_1d == 1 else 'Noise'
    combined_pred_str = 'Signal' if combined_pred == 1 else 'Noise'
    true_label_str = 'Signal' if true_label == 1 else 'Noise'

    # Show if combined prediction is correct
    status = '✓ CORRECT' if combined_pred == true_label else '✗ WRONG'

    print(f"{id_str:<25} {pred_2d_str:<10} {pred_1d_str:<10} {combined_pred_str:<10} "
          f"{true_label_str:<10} {status:<20}")



SAVING DETAILED PREDICTIONS TO CSV
Combined predictions saved to: combined_predictions_by_label.csv

DETAILED PREDICTIONS BY LABEL
Label_ID                  2D         1D         Combined   True       Status              
------------------------------------------------------------------------------------------
HVC192.28-32.50-57        Signal     Noise      Noise      Signal     ✗ WRONG             
HVC193.22-38.75-63        Signal     Noise      Noise      Signal     ✗ WRONG             
HVC194.48-32.49-57        Signal     Noise      Noise      Signal     ✗ WRONG             
HVC195.02-36.10-245       Noise      Noise      Noise      Signal     ✗ WRONG             
HVC195.81-27.19-82        Signal     Signal     Signal     Signal     ✓ CORRECT           
HVC195.91-33.75-57        Signal     Noise      Noise      Signal     ✗ WRONG             
HVC196.18-36.17-75        Noise      Noise      Noise      Signal     ✗ WRONG             
HVC196.58-35.57-89        Noise      Noise      N

In [34]:
# ============================================================================
# SECTION 5: PREDICTION STATISTICS
# ============================================================================

print("\n" + "="*70)
print("PREDICTION STATISTICS")
print("="*70)

# Calculate 2D model statistics
# NEW: Count correct and wrong predictions for 2D model
pred_2d_correct = sum(1 for pred, true in zip(preds_2d, trues_2d) if pred == true)
pred_2d_wrong = sum(1 for pred, true in zip(preds_2d, trues_2d) if pred != true)
pred_2d_total = len(preds_2d)
pred_2d_correct_pct = (pred_2d_correct / pred_2d_total * 100) if pred_2d_total > 0 else 0
pred_2d_wrong_pct = (pred_2d_wrong / pred_2d_total * 100) if pred_2d_total > 0 else 0

print("\n=== 2D MODEL PREDICTIONS ===")
print(f"Total predictions: {pred_2d_total}")
print(f"  ✓ CORRECT: {pred_2d_correct:3d} ({pred_2d_correct_pct:6.2f}%)")
print(f"  ✗ WRONG:   {pred_2d_wrong:3d} ({pred_2d_wrong_pct:6.2f}%)")

# Calculate 1D model statistics
# NEW: Count correct and wrong predictions for 1D model
pred_1d_correct = sum(1 for pred, true in zip(preds_1d, trues_1d) if pred == true)
pred_1d_wrong = sum(1 for pred, true in zip(preds_1d, trues_1d) if pred != true)
pred_1d_total = len(preds_1d)
pred_1d_correct_pct = (pred_1d_correct / pred_1d_total * 100) if pred_1d_total > 0 else 0
pred_1d_wrong_pct = (pred_1d_wrong / pred_1d_total * 100) if pred_1d_total > 0 else 0

print("\n=== 1D MODEL PREDICTIONS ===")
print(f"Total predictions: {pred_1d_total}")
print(f"  ✓ CORRECT: {pred_1d_correct:3d} ({pred_1d_correct_pct:6.2f}%)")
print(f"  ✗ WRONG:   {pred_1d_wrong:3d} ({pred_1d_wrong_pct:6.2f}%)")

# Calculate combined model statistics
# NEW: Count correct and wrong predictions for combined model
pred_combined_correct = sum(1 for pred, true in zip(combined_preds, combined_labels) if pred == true)
pred_combined_wrong = sum(1 for pred, true in zip(combined_preds, combined_labels) if pred != true)
pred_combined_total = len(combined_preds)
pred_combined_correct_pct = (pred_combined_correct / pred_combined_total * 100) if pred_combined_total > 0 else 0
pred_combined_wrong_pct = (pred_combined_wrong / pred_combined_total * 100) if pred_combined_total > 0 else 0

print("\n=== COMBINED MODEL PREDICTIONS ===")
print(f"Total predictions: {pred_combined_total}")
print(f"  ✓ CORRECT: {pred_combined_correct:3d} ({pred_combined_correct_pct:6.2f}%)")
print(f"  ✗ WRONG:   {pred_combined_wrong:3d} ({pred_combined_wrong_pct:6.2f}%)")

# Print comparison summary
print("\n" + "="*70)
print("MODELS COMPARISON SUMMARY")
print("="*70)
print(f"{'Model':<20} {'Accuracy':<15} {'Correct':<20} {'Wrong':<20}")
print("-" * 75)
print(f"{'2D Model':<20} {acc_2d*100:6.2f}%        {pred_2d_correct:3d}/{pred_2d_total:3d} ({pred_2d_correct_pct:5.2f}%) {pred_2d_wrong:3d}/{pred_2d_total:3d} ({pred_2d_wrong_pct:5.2f}%)")
print(f"{'1D Model':<20} {acc_1d*100:6.2f}%        {pred_1d_correct:3d}/{pred_1d_total:3d} ({pred_1d_correct_pct:5.2f}%) {pred_1d_wrong:3d}/{pred_1d_total:3d} ({pred_1d_wrong_pct:5.2f}%)")
print(f"{'Combined Model':<20} {acc_combined*100:6.2f}%        {pred_combined_correct:3d}/{pred_combined_total:3d} ({pred_combined_correct_pct:5.2f}%) {pred_combined_wrong:3d}/{pred_combined_total:3d} ({pred_combined_wrong_pct:5.2f}%)")



PREDICTION STATISTICS

=== 2D MODEL PREDICTIONS ===
Total predictions: 101
  ✓ CORRECT:  63 ( 62.38%)
  ✗ WRONG:    38 ( 37.62%)

=== 1D MODEL PREDICTIONS ===
Total predictions: 101
  ✓ CORRECT:  33 ( 32.67%)
  ✗ WRONG:    68 ( 67.33%)

=== COMBINED MODEL PREDICTIONS ===
Total predictions: 101
  ✓ CORRECT:  32 ( 31.68%)
  ✗ WRONG:    69 ( 68.32%)

MODELS COMPARISON SUMMARY
Model                Accuracy        Correct              Wrong               
---------------------------------------------------------------------------
2D Model              62.38%         63/101 (62.38%)  38/101 (37.62%)
1D Model              32.67%         33/101 (32.67%)  68/101 (67.33%)
Combined Model        31.68%         32/101 (31.68%)  69/101 (68.32%)


In [28]:
# ============================================================================
# FINAL CLEANUP AND SUMMARY
# ============================================================================

print("\n" + "="*70)
print("TRAINING COMPLETED SUCCESSFULLY")
print("="*70)
print(f"Completion Time (UTC): {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"User: Firestar-Reimu")
print(f"Device Used: {device_name}")
print(f"Total Epochs 2D: {num_epochs_2d}, Total Epochs 1D: {num_epochs_1d}")
print(f"Loss Print Interval: Every 5 epochs")
print(f"\nTraining Summary:")
print(f"  2D training samples: {len(arrays_2d_train)}")
print(f"    - False: {len([l for l in labels_2d_train if l == 0])}")
print(f"    - True: {len([l for l in labels_2d_train if l == 1])}")
print(f"  1D training samples: {len(spectra_1d_train)}")
print(f"    - False: {len([l for l in labels_1d_train if l == 0])}")
print(f"    - True: {len([l for l in labels_1d_train if l == 1])}")
print(f"\nTest Summary:")
print(f"  2D test samples: {len(arrays_2d_test)}")
print(f"    - False: {len([l for l in labels_2d_test if l == 0])}")
print(f"    - True: {len([l for l in labels_2d_test if l == 1])}")
print(f"  1D test samples: {len(spectra_1d_test)}")
print(f"    - False: {len([l for l in labels_1d_test if l == 0])}")
print(f"    - True: {len([l for l in labels_1d_test if l == 1])}")
print(f"\nModel Performance:")
print(f"  2D Accuracy:       {acc_2d*100:6.2f}% ({pred_2d_correct}/{pred_2d_total})")
print(f"  1D Accuracy:       {acc_1d*100:6.2f}% ({pred_1d_correct}/{pred_1d_total})")
print(f"  Combined Accuracy: {acc_combined*100:6.2f}% ({pred_combined_correct}/{pred_combined_total})")
print(f"\nModel Files:")
print(f"  2D Model weights: cnn_2d_morph.pt")
print(f"  1D Model weights: cnn_1d_spectrum.pt")
print(f"\nResults:")
print(f"  Predictions CSV: {output_file_combined}")
print("="*70 + "\n")

# Clear GPU memory if CUDA was used
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✓ GPU memory cleared")


TRAINING COMPLETED SUCCESSFULLY
Completion Time (UTC): 2025-11-22 17:40:31
User: Firestar-Reimu
Device Used: NVIDIA GeForce RTX 4060 Ti (GPU)
Total Epochs 2D: 50, Total Epochs 1D: 100
Loss Print Interval: Every 5 epochs

Training Summary:
  2D training samples: 38
    - False: 14
    - True: 24
  1D training samples: 38
    - False: 14
    - True: 24

Test Summary:
  2D test samples: 101
    - False: 11
    - True: 90
  1D test samples: 101
    - False: 11
    - True: 90

Model Performance:
  2D Accuracy:        62.38% (63/101)
  1D Accuracy:        79.21% (80/101)
  Combined Accuracy:  55.45% (56/101)

Model Files:
  2D Model weights: cnn_2d_morph.pt
  1D Model weights: cnn_1d_spectrum.pt

Results:
  Predictions CSV: combined_predictions_by_label.csv

✓ GPU memory cleared
